# Flan-T5 LoRA Training on PubMedQA

This notebook demonstrates:
1. Model loading and inspection
2. LoRA (Low-Rank Adaptation) implementation for efficient fine-tuning
3. Training on PubMedQA dataset
4. Evaluation and inference

**Environment**: Kaggle with T4 GPU

## 1. Setup & Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TF warnings
import warnings
warnings.filterwarnings('ignore')

import re
import math
from collections import defaultdict, Counter
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import DataCollatorForSeq2Seq

from rouge_score import rouge_scorer
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

2026-01-16 10:04:22.123550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768557862.298603      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768557862.351055      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768557862.762894      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768557862.762936      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768557862.762939      55 computation_placer.cc:177] computation placer alr

## 2. Utility Functions

In [4]:
class LoRALinear(nn.Module):
    """LoRA wrapper for linear layers."""
    
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False  # Freeze base

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        # Get the device of the base layer
        device = base_linear.weight.device

        self.A = nn.Linear(in_dim, r, bias=False, device=device)
        self.B = nn.Linear(r, out_dim, bias=False, device=device)
        self.scaling = alpha / r

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias to satisfy HF generate()
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

In [5]:
def freeze_all_params(model):
    """Freeze all model parameters."""
    for p in model.parameters():
        p.requires_grad = False


def enable_ffn_training(model):
    """Enable training only for FFN (DenseReluDense) layers in both encoder and decoder."""
    for name, module in model.named_modules():
        if module.__class__.__name__ == "T5DenseGatedActDense":
            for p in module.parameters():
                p.requires_grad = True


def apply_lora_to_ffn(model, r=8, alpha=1.0):
    """Apply LoRA to FFN layers (T5DenseGatedActDense)."""
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            continue  # Only wrap higher-level FFN
        if module.__class__.__name__ == "T5DenseGatedActDense":
            module.wi_0 = LoRALinear(module.wi_0, r=r, alpha=alpha)
            module.wi_1 = LoRALinear(module.wi_1, r=r, alpha=alpha)
            module.wo = LoRALinear(module.wo, r=r, alpha=alpha)


def merge_lora_to_linear(model):
    """Merge LoRA weights back into base linear layers."""
    for name, module in model.named_modules():
        if isinstance(module, LoRALinear):
            new_linear = nn.Linear(
                module.base.in_features,
                module.base.out_features,
                bias=(module.base.bias is not None)
            ).to(module.base.weight.device)

            new_linear.weight.data = module.base.weight.data + (module.B.weight @ module.A.weight) * module.scaling

            if module.base.bias is not None:
                new_linear.bias.data = module.base.bias.data

            parent = module._modules
            for key, child in parent.items():
                if child is module:
                    parent[key] = new_linear
                    break


def enable_lora_forward(model):
    """Temporarily compute effective weights for generation."""
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module._original_forward = module.forward
            module.forward = lambda x, m=module: m.base(x) + m.scaling * m.B(m.A(x))


def disable_lora_forward(model):
    """Restore original forward for training."""
    for module in model.modules():
        if isinstance(module, LoRALinear) and hasattr(module, "_original_forward"):
            module.forward = module._original_forward
            del module._original_forward

In [ ]:
def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def print_trainable_params(model):
    """Print names of trainable parameters."""
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name)


def print_param_stats(model):
    """Print parameter statistics by module."""
    param_stats = defaultdict(int)
    for name, param in model.named_parameters():
        param_stats[name.split('.')[0]] += param.numel()
    for k, v in param_stats.items():
        print(f"{k}: {v:,}")


def extract_explanation(text):
    """Extract explanation from generated text."""
    match = re.search(r"Explanation:\s*(.+)", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r"Answer:\s*(?:yes|no|maybe)[.,]?\s*(.+)", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return text


def extract_short_answer(text):
    """Extract short answer (yes/no/maybe) from generated text."""
    match = re.search(r"Answer:\s*(yes|no|maybe)", text, re.IGNORECASE)
    return match.group(1).lower() if match else "unknown"


def compute_rouge_scores(predictions, references, scorer):
    """Compute ROUGE scores with detailed per-example breakdown."""
    results = {
        'rouge1': [], 'rouge2': [], 'rougeL': [],
        'rouge1_p': [], 'rouge1_r': [],
        'rouge2_p': [], 'rouge2_r': [],
        'rougeL_p': [], 'rougeL_r': []
    }
    
    for pred, ref in zip(predictions, references):
        pred_expl = extract_explanation(pred)
        ref_expl = extract_explanation(ref)
        
        scores = scorer.score(ref_expl, pred_expl)
        results['rouge1'].append(scores['rouge1'].fmeasure)
        results['rouge2'].append(scores['rouge2'].fmeasure)
        results['rougeL'].append(scores['rougeL'].fmeasure)
        results['rouge1_p'].append(scores['rouge1'].precision)
        results['rouge1_r'].append(scores['rouge1'].recall)
        results['rouge2_p'].append(scores['rouge2'].precision)
        results['rouge2_r'].append(scores['rouge2'].recall)
        results['rougeL_p'].append(scores['rougeL'].precision)
        results['rougeL_r'].append(scores['rougeL'].recall)
    
    return {k: np.mean(v) if v else 0 for k, v in results.items()}, results


def compute_all_metrics(generated_texts, val_short_answers, val_raw_outputs, scorer):
    """Compute comprehensive evaluation metrics."""
    # Extract predictions
    pred_labels = [extract_short_answer(t) for t in generated_texts]
    true_labels = [a.lower() for a in val_short_answers]
    
    # Short answer metrics
    correct = sum(1 for p, t in zip(pred_labels, true_labels) if p == t)
    accuracy = correct / len(true_labels)
    
    # Per-class metrics
    labels = ['yes', 'no', 'maybe']
    
    # Filter for valid predictions only
    valid_mask = [p in labels for p in pred_labels]
    valid_preds = [p for p, v in zip(pred_labels, valid_mask) if v]
    valid_trues = [t for t, v in zip(true_labels, valid_mask) if v]
    
    # Classification metrics
    if valid_preds:
        f1_macro = f1_score(valid_trues, valid_preds, labels=labels, average='macro', zero_division=0)
        f1_weighted = f1_score(valid_trues, valid_preds, labels=labels, average='weighted', zero_division=0)
        precision_macro = precision_score(valid_trues, valid_preds, labels=labels, average='macro', zero_division=0)
        recall_macro = recall_score(valid_trues, valid_preds, labels=labels, average='macro', zero_division=0)
        
        f1_per_class = f1_score(valid_trues, valid_preds, labels=labels, average=None, zero_division=0)
        precision_per_class = precision_score(valid_trues, valid_preds, labels=labels, average=None, zero_division=0)
        recall_per_class = recall_score(valid_trues, valid_preds, labels=labels, average=None, zero_division=0)
    else:
        f1_macro = f1_weighted = precision_macro = recall_macro = 0
        f1_per_class = precision_per_class = recall_per_class = [0, 0, 0]
    
    # ROUGE scores
    rouge_avg, rouge_detailed = compute_rouge_scores(generated_texts, val_raw_outputs, scorer)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_per_class': dict(zip(labels, f1_per_class)),
        'precision_per_class': dict(zip(labels, precision_per_class)),
        'recall_per_class': dict(zip(labels, recall_per_class)),
        'rouge_avg': rouge_avg,
        'rouge_detailed': rouge_detailed,
        'pred_labels': pred_labels,
        'true_labels': true_labels,
        'valid_preds': valid_preds,
        'valid_trues': valid_trues
    }

In [7]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

print(f"Model loaded on: {device}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on: cuda


### 3.1 Model Architecture Inspection

In [10]:
print("Encoder Block 0:")
print(model.encoder.block[0])

Encoder Block 0:
T5Block(
  (layer): ModuleList(
    (0): T5LayerSelfAttention(
      (SelfAttention): T5Attention(
        (q): Linear(in_features=768, out_features=768, bias=False)
        (k): Linear(in_features=768, out_features=768, bias=False)
        (v): Linear(in_features=768, out_features=768, bias=False)
        (o): Linear(in_features=768, out_features=768, bias=False)
        (relative_attention_bias): Embedding(32, 12)
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): T5LayerFF(
      (DenseReluDense): T5DenseGatedActDense(
        (wi_0): Linear(in_features=768, out_features=2048, bias=False)
        (wi_1): Linear(in_features=768, out_features=2048, bias=False)
        (wo): Linear(in_features=2048, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (act): NewGELUActivation()
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)


In [12]:
print("LM Head:")
print(model.lm_head)

LM Head:
Linear(in_features=768, out_features=32128, bias=False)


In [14]:
print("\nParameter statistics by module:")
print_param_stats(model)


Parameter statistics by module:
shared: 24,674,304
encoder: 84,954,240
decoder: 113,275,008
lm_head: 24,674,304


## 4. Dataset Preparation (PubMedQA)

In [ ]:
# Preprocessing parameters
max_input_length = 512
max_output_length = 128


def preprocess_pubmedqa(example):
    """Preprocess PubMedQA examples into input-output format."""
    context = " ".join(example["context"]["contexts"])
    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: Answer yes, no, or maybe. Then justify your answer."
    )
    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )
    short_answer = example['final_decision']

    return {
        "input": input_text,
        "output": target_text,
        "short_answer": short_answer
    }


def tokenize_for_t5(example):
    """Tokenize examples for T5 training."""
    input_enc = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=max_input_length,
    )

    target_enc = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=max_output_length,
    )

    labels = target_enc["input_ids"]
    labels = [l if l != tokenizer.pad_token_id else -100 for l in labels]

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": labels,
    }


def collate_fn(batch):
    """Collate function for DataLoader."""
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    attention_mask = torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [18]:
# Load dataset
dataset = load_dataset("pubmed_qa", "pqa_labeled")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


In [20]:
print("\nSample example:")
print(dataset["train"][0])


Sample example:
{'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PC

In [22]:
# Preprocess dataset
dataset = dataset.map(
    preprocess_pubmedqa,
    remove_columns=dataset["train"].column_names
)
print("\nPreprocessed example:")
print(dataset["train"][0])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Preprocessed example:
{'input': 'Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early sta

In [ ]:
# Split into train and validation sets
split_datasets = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_data = split_datasets["train"]
val_data = split_datasets["test"]

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")

In [24]:
# Store validation short answers for evaluation
val_short_answers = [ex["short_answer"] for ex in val_data]
val_raw_inputs = [ex["input"] for ex in val_data]
val_raw_outputs = [ex["output"] for ex in val_data]

# Remove short_answer column before tokenization
train_data = train_data.remove_columns("short_answer")
val_data = val_data.remove_columns("short_answer")

print(f"Train columns: {train_data.column_names}")
print(f"Val columns: {val_data.column_names}")

Train columns: ['input', 'output']
Val columns: ['input', 'output']


In [ ]:
# Tokenize datasets
train_dataset = train_data.map(tokenize_for_t5, remove_columns=["input", "output"])
val_dataset = val_data.map(tokenize_for_t5, remove_columns=["input", "output"])

print(f"Train dataset columns: {train_dataset.column_names}")
print(f"Val dataset columns: {val_dataset.column_names}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

In [26]:
# Create DataLoaders
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 113
Val batches: 13


## 5. Apply LoRA and Prepare for Training

In [29]:
# LoRA hyperparameters
rank = 128
alpha = 256

# Freeze all parameters
freeze_all_params(model)

# Apply LoRA to FFN layers
apply_lora_to_ffn(model, r=rank, alpha=alpha)

# Make only LoRA parameters trainable
for name, param in model.named_parameters():
    if "A.weight" in name or "B.weight" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

print(f"LoRA applied with rank={rank}, alpha={alpha}")

LoRA applied with rank=128, alpha=256


In [ ]:
# Training hyperparameters
num_epochs = 100
learning_rate = 1e-4
eval_every = 10  # Evaluate every N epochs

# Setup optimizer (only for trainable parameters)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=learning_rate)

# Print trainable parameter count
total_params, trainable = count_parameters(model)
print(f"📊 Total parameters: {total_params:,}")
print(f"🎯 Trainable parameters: {trainable:,} ({100 * trainable / total_params:.2f}%)")
print(f"⚙️ Optimizer: AdamW (lr={learning_rate})")
print(f"📅 Epochs: {num_epochs}")
print(f"📈 Evaluation every: {eval_every} epochs")

## 6. Training Loop with Evaluation

In [ ]:
# Initialize ROUGE scorer and training history
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Training history for visualization
training_history = {
    'epoch': [],
    'train_loss': [],
    'accuracy': [],
    'f1_macro': [],
    'rouge1': [],
    'rouge2': [],
    'rougeL': []
}

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    epoch_losses = []
    loop = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Epoch {epoch+1}")

    for step, batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=1.0
        )
        
        optimizer.step()

        running_loss += loss.item()
        epoch_losses.append(loss.item())
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"\n📊 Epoch {epoch+1}/{num_epochs} Training Loss: {avg_loss:.4f}")

    # Evaluation phase (only every eval_every epochs)
    if (epoch + 1) % eval_every == 0 or (epoch + 1) == num_epochs:
        print(f"\n🔍 Running evaluation at epoch {epoch+1}...")
        model.eval()
        enable_lora_forward(model)

        all_generated_texts = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="🔍 Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=128,
                    do_sample=False
                )

                gen_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
                all_generated_texts.extend(gen_texts)

        # Compute all metrics
        metrics = compute_all_metrics(all_generated_texts, val_short_answers, val_raw_outputs, scorer)
        
        # Store history
        training_history['epoch'].append(epoch + 1)
        training_history['train_loss'].append(avg_loss)
        training_history['accuracy'].append(metrics['accuracy'])
        training_history['f1_macro'].append(metrics['f1_macro'])
        training_history['rouge1'].append(metrics['rouge_avg']['rouge1'])
        training_history['rouge2'].append(metrics['rouge_avg']['rouge2'])
        training_history['rougeL'].append(metrics['rouge_avg']['rougeL'])
        
        # Print summary
        print(f"\n{'='*60}")
        print(f"📈 Epoch {epoch+1} Evaluation Summary")
        print(f"{'='*60}")
        print(f"✅ Short-Answer Accuracy: {metrics['accuracy']:.4f}")
        print(f"🎯 F1-Score (Macro): {metrics['f1_macro']:.4f}")
        print(f"📝 ROUGE-1: {metrics['rouge_avg']['rouge1']:.4f} | ROUGE-2: {metrics['rouge_avg']['rouge2']:.4f} | ROUGE-L: {metrics['rouge_avg']['rougeL']:.4f}")
        print(f"{'='*60}\n")

        disable_lora_forward(model)

    else:
        # Store only training loss for non-evaluation epochs
        training_history['epoch'].append(epoch + 1)
        training_history['train_loss'].append(avg_loss)
        
        # Use previous metrics or 0 for non-eval epochs
        if training_history['accuracy']:
            training_history['accuracy'].append(training_history['accuracy'][-1])
            training_history['f1_macro'].append(training_history['f1_macro'][-1])
            training_history['rouge1'].append(training_history['rouge1'][-1])
            training_history['rouge2'].append(training_history['rouge2'][-1])
            training_history['rougeL'].append(training_history['rougeL'][-1])
        else:
            training_history['accuracy'].append(0)
            training_history['f1_macro'].append(0)
            training_history['rouge1'].append(0)
            training_history['rouge2'].append(0)
            training_history['rougeL'].append(0)

print("\n✅ Training Complete!")

Epoch 1: 100%|██████████| 113/113 [02:24<00:00,  1.28s/it, loss=2.38]


Epoch 1 average loss: 2.1627


Evaluating: 100%|██████████| 13/13 [00:21<00:00,  1.64s/it]


Epoch 1 short-answer accuracy: 0.4900
Epoch 1 ROUGE-1: 0.2942, ROUGE-2: 0.1175, ROUGE-L: 0.2318


Epoch 2: 100%|██████████| 113/113 [02:32<00:00,  1.35s/it, loss=1.87]


...

Epoch 2 short-answer accuracy: 0.5300
Epoch 2 ROUGE-1: 0.3152, ROUGE-2: 0.1261, ROUGE-L: 0.2462


Epoch 3: 100%|██████████| 113/113 [02:32<00:00,  1.35s/it, loss=1.48]


Epoch 3 average loss: 1.8097


Evaluating: 100%|██████████| 13/13 [00:26<00:00,  2.02s/it]

Epoch 3 short-answer accuracy: 0.5200
Epoch 3 ROUGE-1: 0.3128, ROUGE-2: 0.1193, ROUGE-L: 0.2395


In [ ]:
# ============================================================================
# 📊 COMPREHENSIVE EVALUATION & VISUALIZATION
# ============================================================================

# Enable LoRA for final evaluation
enable_lora_forward(model)
model.eval()

print("🔄 Generating predictions on validation set...")
generated_texts = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Generating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False
        )

        batch_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
        generated_texts.extend(batch_texts)

# Compute comprehensive metrics
print("📊 Computing evaluation metrics...")
final_metrics = compute_all_metrics(generated_texts, val_short_answers, val_raw_outputs, scorer)

disable_lora_forward(model)

Generating on val set: 100%|██████████| 13/13 [00:26<00:00,  2.03s/it]


Final Evaluation Metrics:
Short-answer Accuracy: 0.5200
ROUGE-1: 0.3128
ROUGE-2: 0.1193
ROUGE-L: 0.2395

Sample Generations:

Example 1:
Question + Context + Instruction:
Question: Is eligibility for a chemotherapy protocol a good prognostic factor for invasive bladder cancer after radical cystectomy? Context: To assess whether eligibility to an adjuvant chemotherapy p...

True Answer: Answer: yes. Explanation: These data suggest that being willing and fit enough for a chemotherapy protocol is a good prognostic factor for invasive bladder cancer. This eligibility bias emphasizes the need for prospective, randomized trials, and indicates that single-group studies using historical or matched controls have to be interpreted with caution.
Generated Answer:
Answer: yes. Explanation: The eligibility to cisplatin monotherapy after cystectomy is a good prognostic factor for invasive bladder cancer.

ROUGE-1: 0.3692, ROUGE-2: 0.2540, ROUGE-L: 0.2769
--------------------------------------------

In [ ]:
# ============================================================================
# 📈 TRAINING PROGRESS VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🎓 Training Progress Dashboard', fontsize=16, fontweight='bold', y=1.02)

epochs = training_history['epoch']

# Plot 1: Training Loss
ax1 = axes[0, 0]
ax1.plot(epochs, training_history['train_loss'], 'b-o', linewidth=2, markersize=8, label='Training Loss')
ax1.fill_between(epochs, training_history['train_loss'], alpha=0.3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('📉 Training Loss Over Epochs', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
for i, v in enumerate(training_history['train_loss']):
    ax1.annotate(f'{v:.3f}', (epochs[i], v), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

# Plot 2: Accuracy & F1
ax2 = axes[0, 1]
ax2.plot(epochs, training_history['accuracy'], 'g-o', linewidth=2, markersize=8, label='Accuracy')
ax2.plot(epochs, training_history['f1_macro'], 'r-s', linewidth=2, markersize=8, label='F1 (Macro)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('🎯 Classification Metrics Over Epochs', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

# Plot 3: ROUGE Scores
ax3 = axes[1, 0]
ax3.plot(epochs, training_history['rouge1'], 'c-o', linewidth=2, markersize=8, label='ROUGE-1')
ax3.plot(epochs, training_history['rouge2'], 'm-s', linewidth=2, markersize=8, label='ROUGE-2')
ax3.plot(epochs, training_history['rougeL'], 'y-^', linewidth=2, markersize=8, label='ROUGE-L')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Score')
ax3.set_title('📝 ROUGE Scores Over Epochs', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 1)

# Plot 4: Final Metrics Summary Bar Chart
ax4 = axes[1, 1]
final_epoch_metrics = {
    'Accuracy': training_history['accuracy'][-1],
    'F1 Macro': training_history['f1_macro'][-1],
    'ROUGE-1': training_history['rouge1'][-1],
    'ROUGE-2': training_history['rouge2'][-1],
    'ROUGE-L': training_history['rougeL'][-1]
}
colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6', '#f39c12']
bars = ax4.bar(final_epoch_metrics.keys(), final_epoch_metrics.values(), color=colors, edgecolor='black', linewidth=1.2)
ax4.set_ylabel('Score')
ax4.set_title('📊 Final Epoch Metrics Summary', fontweight='bold')
ax4.set_ylim(0, 1)
for bar, val in zip(bars, final_epoch_metrics.values()):
    ax4.annotate(f'{val:.3f}', (bar.get_x() + bar.get_width()/2, bar.get_height()), 
                 textcoords="offset points", xytext=(0,5), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 🎯 CLASSIFICATION METRICS VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('🎯 Short-Answer Classification Analysis', fontsize=16, fontweight='bold', y=1.02)

# Confusion Matrix
ax1 = axes[0]
labels = ['yes', 'no', 'maybe']
cm = confusion_matrix(final_metrics['valid_trues'], final_metrics['valid_preds'], labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, 
            ax=ax1, annot_kws={'size': 14, 'weight': 'bold'}, linewidths=0.5)
ax1.set_xlabel('Predicted Label', fontweight='bold')
ax1.set_ylabel('True Label', fontweight='bold')
ax1.set_title('🔢 Confusion Matrix', fontweight='bold', fontsize=12)

# Per-Class F1, Precision, Recall
ax2 = axes[1]
x = np.arange(len(labels))
width = 0.25

f1_vals = [final_metrics['f1_per_class'][l] for l in labels]
prec_vals = [final_metrics['precision_per_class'][l] for l in labels]
rec_vals = [final_metrics['recall_per_class'][l] for l in labels]

bars1 = ax2.bar(x - width, f1_vals, width, label='F1-Score', color='#3498db', edgecolor='black')
bars2 = ax2.bar(x, prec_vals, width, label='Precision', color='#2ecc71', edgecolor='black')
bars3 = ax2.bar(x + width, rec_vals, width, label='Recall', color='#e74c3c', edgecolor='black')

ax2.set_xlabel('Class', fontweight='bold')
ax2.set_ylabel('Score', fontweight='bold')
ax2.set_title('📊 Per-Class Metrics', fontweight='bold', fontsize=12)
ax2.set_xticks(x)
ax2.set_xticklabels([l.upper() for l in labels])
ax2.legend(loc='upper right')
ax2.set_ylim(0, 1.1)
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax2.annotate(f'{height:.2f}', (bar.get_x() + bar.get_width()/2, height),
                        textcoords="offset points", xytext=(0,3), ha='center', fontsize=8)

# Label Distribution
ax3 = axes[2]
pred_counts = Counter(final_metrics['pred_labels'])
true_counts = Counter(final_metrics['true_labels'])

all_labels = ['yes', 'no', 'maybe', 'unknown']
pred_values = [pred_counts.get(l, 0) for l in all_labels]
true_values = [true_counts.get(l, 0) for l in all_labels]

x = np.arange(len(all_labels))
width = 0.35

bars1 = ax3.bar(x - width/2, true_values, width, label='True Labels', color='#2ecc71', edgecolor='black')
bars2 = ax3.bar(x + width/2, pred_values, width, label='Predicted Labels', color='#3498db', edgecolor='black')

ax3.set_xlabel('Label', fontweight='bold')
ax3.set_ylabel('Count', fontweight='bold')
ax3.set_title('📈 Label Distribution', fontweight='bold', fontsize=12)
ax3.set_xticks(x)
ax3.set_xticklabels([l.upper() for l in all_labels])
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax3.annotate(f'{int(height)}', (bar.get_x() + bar.get_width()/2, height),
                        textcoords="offset points", xytext=(0,3), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 📝 ROUGE SCORES DETAILED ANALYSIS
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📝 Long-Answer ROUGE Analysis', fontsize=16, fontweight='bold', y=1.02)

rouge_detailed = final_metrics['rouge_detailed']

# Plot 1: ROUGE Score Distributions (Box Plot)
ax1 = axes[0, 0]
rouge_data = pd.DataFrame({
    'ROUGE-1': rouge_detailed['rouge1'],
    'ROUGE-2': rouge_detailed['rouge2'],
    'ROUGE-L': rouge_detailed['rougeL']
})
box = ax1.boxplot([rouge_data['ROUGE-1'], rouge_data['ROUGE-2'], rouge_data['ROUGE-L']], 
                   labels=['ROUGE-1', 'ROUGE-2', 'ROUGE-L'], patch_artist=True)
colors = ['#3498db', '#9b59b6', '#f39c12']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Score')
ax1.set_title('📦 ROUGE Score Distribution', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add mean markers
means = [np.mean(rouge_detailed['rouge1']), np.mean(rouge_detailed['rouge2']), np.mean(rouge_detailed['rougeL'])]
ax1.scatter([1, 2, 3], means, color='red', marker='D', s=50, zorder=5, label='Mean')
ax1.legend()

# Plot 2: ROUGE Histograms
ax2 = axes[0, 1]
ax2.hist(rouge_detailed['rouge1'], bins=20, alpha=0.7, label='ROUGE-1', color='#3498db', edgecolor='black')
ax2.hist(rouge_detailed['rouge2'], bins=20, alpha=0.7, label='ROUGE-2', color='#9b59b6', edgecolor='black')
ax2.hist(rouge_detailed['rougeL'], bins=20, alpha=0.7, label='ROUGE-L', color='#f39c12', edgecolor='black')
ax2.set_xlabel('Score')
ax2.set_ylabel('Frequency')
ax2.set_title('📊 ROUGE Score Histogram', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Precision vs Recall for ROUGE-1
ax3 = axes[1, 0]
ax3.scatter(rouge_detailed['rouge1_p'], rouge_detailed['rouge1_r'], alpha=0.5, c='#3498db', edgecolor='black', s=50)
ax3.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='P=R line')
ax3.set_xlabel('Precision')
ax3.set_ylabel('Recall')
ax3.set_title('🎯 ROUGE-1 Precision vs Recall', fontweight='bold')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.legend()
ax3.grid(True, alpha=0.3)

# Add mean point
mean_p = np.mean(rouge_detailed['rouge1_p'])
mean_r = np.mean(rouge_detailed['rouge1_r'])
ax3.scatter([mean_p], [mean_r], color='red', marker='*', s=200, zorder=5, label=f'Mean ({mean_p:.2f}, {mean_r:.2f})')
ax3.legend()

# Plot 4: ROUGE Metrics Summary with Precision/Recall breakdown
ax4 = axes[1, 1]
metrics_summary = {
    'ROUGE-1\nF1': final_metrics['rouge_avg']['rouge1'],
    'ROUGE-1\nPrecision': final_metrics['rouge_avg']['rouge1_p'],
    'ROUGE-1\nRecall': final_metrics['rouge_avg']['rouge1_r'],
    'ROUGE-2\nF1': final_metrics['rouge_avg']['rouge2'],
    'ROUGE-L\nF1': final_metrics['rouge_avg']['rougeL'],
}
colors = ['#3498db', '#2980b9', '#1abc9c', '#9b59b6', '#f39c12']
bars = ax4.bar(metrics_summary.keys(), metrics_summary.values(), color=colors, edgecolor='black', linewidth=1.2)
ax4.set_ylabel('Score')
ax4.set_title('📋 ROUGE Metrics Breakdown', fontweight='bold')
ax4.set_ylim(0, 1)
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, metrics_summary.values()):
    ax4.annotate(f'{val:.3f}', (bar.get_x() + bar.get_width()/2, bar.get_height()), 
                 textcoords="offset points", xytext=(0,5), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 📋 COMPREHENSIVE METRICS SUMMARY TABLE
# ============================================================================

print("\n" + "="*80)
print("📊 COMPREHENSIVE EVALUATION REPORT")
print("="*80)

# Create summary DataFrame
summary_data = {
    'Metric Category': [],
    'Metric Name': [],
    'Value': [],
    'Description': []
}

# Classification Metrics
classification_metrics = [
    ('Classification', 'Accuracy', f"{final_metrics['accuracy']:.4f}", 'Overall correct predictions'),
    ('Classification', 'F1-Score (Macro)', f"{final_metrics['f1_macro']:.4f}", 'Balanced F1 across classes'),
    ('Classification', 'F1-Score (Weighted)', f"{final_metrics['f1_weighted']:.4f}", 'Weighted by class frequency'),
    ('Classification', 'Precision (Macro)', f"{final_metrics['precision_macro']:.4f}", 'Avg precision across classes'),
    ('Classification', 'Recall (Macro)', f"{final_metrics['recall_macro']:.4f}", 'Avg recall across classes'),
]

# Per-class metrics
for label in ['yes', 'no', 'maybe']:
    classification_metrics.extend([
        ('Per-Class', f'F1 ({label.upper()})', f"{final_metrics['f1_per_class'][label]:.4f}", f'F1 for {label} class'),
        ('Per-Class', f'Precision ({label.upper()})', f"{final_metrics['precision_per_class'][label]:.4f}", f'Precision for {label} class'),
        ('Per-Class', f'Recall ({label.upper()})', f"{final_metrics['recall_per_class'][label]:.4f}", f'Recall for {label} class'),
    ])

# ROUGE Metrics
rouge_metrics = [
    ('ROUGE', 'ROUGE-1 (F1)', f"{final_metrics['rouge_avg']['rouge1']:.4f}", 'Unigram overlap F1'),
    ('ROUGE', 'ROUGE-1 (Precision)', f"{final_metrics['rouge_avg']['rouge1_p']:.4f}", 'Unigram precision'),
    ('ROUGE', 'ROUGE-1 (Recall)', f"{final_metrics['rouge_avg']['rouge1_r']:.4f}", 'Unigram recall'),
    ('ROUGE', 'ROUGE-2 (F1)', f"{final_metrics['rouge_avg']['rouge2']:.4f}", 'Bigram overlap F1'),
    ('ROUGE', 'ROUGE-2 (Precision)', f"{final_metrics['rouge_avg']['rouge2_p']:.4f}", 'Bigram precision'),
    ('ROUGE', 'ROUGE-2 (Recall)', f"{final_metrics['rouge_avg']['rouge2_r']:.4f}", 'Bigram recall'),
    ('ROUGE', 'ROUGE-L (F1)', f"{final_metrics['rouge_avg']['rougeL']:.4f}", 'Longest common subsequence F1'),
    ('ROUGE', 'ROUGE-L (Precision)', f"{final_metrics['rouge_avg']['rougeL_p']:.4f}", 'LCS precision'),
    ('ROUGE', 'ROUGE-L (Recall)', f"{final_metrics['rouge_avg']['rougeL_r']:.4f}", 'LCS recall'),
]

all_metrics = classification_metrics + rouge_metrics

for cat, name, val, desc in all_metrics:
    summary_data['Metric Category'].append(cat)
    summary_data['Metric Name'].append(name)
    summary_data['Value'].append(val)
    summary_data['Description'].append(desc)

summary_df = pd.DataFrame(summary_data)

# Style the DataFrame
def color_metrics(val):
    try:
        num = float(val)
        if num >= 0.7:
            return 'background-color: #d4edda; color: #155724'
        elif num >= 0.5:
            return 'background-color: #fff3cd; color: #856404'
        else:
            return 'background-color: #f8d7da; color: #721c24'
    except:
        return ''

styled_df = summary_df.style.applymap(color_metrics, subset=['Value'])
display(styled_df)

# Print key highlights
print("\n" + "="*80)
print("🌟 KEY HIGHLIGHTS")
print("="*80)
print(f"""
┌─────────────────────────────────────────────────────────────┐
│  📊 SHORT-ANSWER CLASSIFICATION                             │
├─────────────────────────────────────────────────────────────┤
│  ✅ Accuracy:           {final_metrics['accuracy']:.2%}                              │
│  🎯 F1-Score (Macro):   {final_metrics['f1_macro']:.2%}                              │
│  📈 Precision (Macro):  {final_metrics['precision_macro']:.2%}                              │
│  📉 Recall (Macro):     {final_metrics['recall_macro']:.2%}                              │
├─────────────────────────────────────────────────────────────┤
│  📝 LONG-ANSWER QUALITY (ROUGE SCORES)                      │
├─────────────────────────────────────────────────────────────┤
│  📖 ROUGE-1 (F1):       {final_metrics['rouge_avg']['rouge1']:.2%}                              │
│  📖 ROUGE-2 (F1):       {final_metrics['rouge_avg']['rouge2']:.2%}                              │
│  📖 ROUGE-L (F1):       {final_metrics['rouge_avg']['rougeL']:.2%}                              │
└─────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================================================
# 🔍 SAMPLE PREDICTIONS WITH DETAILED ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("🔍 SAMPLE PREDICTIONS ANALYSIS")
print("="*80)

# Select diverse examples (one from each class if possible)
examples_to_show = []
for target_label in ['yes', 'no', 'maybe']:
    for i, true_label in enumerate(final_metrics['true_labels']):
        if true_label == target_label and len(examples_to_show) < 5:
            if i not in [e[0] for e in examples_to_show]:
                examples_to_show.append((i, true_label))
                break

# Add a couple more random examples
import random
random.seed(42)
remaining = [i for i in range(len(generated_texts)) if i not in [e[0] for e in examples_to_show]]
for idx in random.sample(remaining, min(2, len(remaining))):
    examples_to_show.append((idx, final_metrics['true_labels'][idx]))

for example_num, (idx, true_label) in enumerate(examples_to_show[:5], 1):
    pred_label = final_metrics['pred_labels'][idx]
    is_correct = pred_label == true_label
    
    # Compute individual ROUGE
    individual_rouge = scorer.score(
        extract_explanation(val_raw_outputs[idx]), 
        extract_explanation(generated_texts[idx])
    )
    
    # Display
    status_emoji = "✅" if is_correct else "❌"
    
    print(f"\n{'─'*80}")
    print(f"📌 EXAMPLE {example_num} {status_emoji}")
    print(f"{'─'*80}")
    
    # Truncate question for display
    question_preview = val_raw_inputs[idx][:300] + "..." if len(val_raw_inputs[idx]) > 300 else val_raw_inputs[idx]
    
    print(f"\n📋 QUESTION (truncated):")
    print(f"   {question_preview}")
    
    print(f"\n🎯 TRUE ANSWER:")
    print(f"   Short: {true_label.upper()}")
    true_expl = extract_explanation(val_raw_outputs[idx])[:200] + "..." if len(extract_explanation(val_raw_outputs[idx])) > 200 else extract_explanation(val_raw_outputs[idx])
    print(f"   Explanation: {true_expl}")
    
    print(f"\n🤖 MODEL PREDICTION:")
    print(f"   Short: {pred_label.upper()} {'✓' if is_correct else '✗'}")
    pred_expl = extract_explanation(generated_texts[idx])[:200] + "..." if len(extract_explanation(generated_texts[idx])) > 200 else extract_explanation(generated_texts[idx])
    print(f"   Explanation: {pred_expl}")
    
    print(f"\n📊 ROUGE SCORES FOR THIS EXAMPLE:")
    print(f"   ├── ROUGE-1: F1={individual_rouge['rouge1'].fmeasure:.3f} | P={individual_rouge['rouge1'].precision:.3f} | R={individual_rouge['rouge1'].recall:.3f}")
    print(f"   ├── ROUGE-2: F1={individual_rouge['rouge2'].fmeasure:.3f} | P={individual_rouge['rouge2'].precision:.3f} | R={individual_rouge['rouge2'].recall:.3f}")
    print(f"   └── ROUGE-L: F1={individual_rouge['rougeL'].fmeasure:.3f} | P={individual_rouge['rougeL'].precision:.3f} | R={individual_rouge['rougeL'].recall:.3f}")

print(f"\n{'='*80}")

In [ ]:
# ============================================================================
# 📈 FINAL COMPREHENSIVE DASHBOARD
# ============================================================================

fig = plt.figure(figsize=(16, 12))
fig.suptitle('🏆 Final Model Evaluation Dashboard', fontsize=18, fontweight='bold', y=0.98)

# Create grid
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. Radar Chart for Overall Metrics
ax1 = fig.add_subplot(gs[0, 0], projection='polar')
categories = ['Accuracy', 'F1 Macro', 'Precision', 'Recall', 'ROUGE-1', 'ROUGE-L']
values = [
    final_metrics['accuracy'],
    final_metrics['f1_macro'],
    final_metrics['precision_macro'],
    final_metrics['recall_macro'],
    final_metrics['rouge_avg']['rouge1'],
    final_metrics['rouge_avg']['rougeL']
]
values += values[:1]  # Complete the loop
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

ax1.plot(angles, values, 'o-', linewidth=2, color='#3498db')
ax1.fill(angles, values, alpha=0.25, color='#3498db')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, size=8)
ax1.set_ylim(0, 1)
ax1.set_title('🎯 Overall Performance', fontweight='bold', size=11, pad=15)

# 2. Gauge-style metrics
ax2 = fig.add_subplot(gs[0, 1])
ax2.axis('off')

# Create visual metric cards
metrics_display = [
    ('🎯 Accuracy', final_metrics['accuracy'], '#2ecc71'),
    ('📊 F1 Score', final_metrics['f1_macro'], '#3498db'),
    ('📝 ROUGE-1', final_metrics['rouge_avg']['rouge1'], '#9b59b6'),
    ('📖 ROUGE-L', final_metrics['rouge_avg']['rougeL'], '#f39c12'),
]

for i, (name, value, color) in enumerate(metrics_display):
    y_pos = 0.85 - i * 0.22
    # Metric name
    ax2.text(0.1, y_pos, name, fontsize=11, fontweight='bold', transform=ax2.transAxes)
    # Progress bar background
    ax2.barh(y_pos - 0.05, 1, height=0.08, color='#ecf0f1', transform=ax2.transAxes)
    # Progress bar value
    ax2.barh(y_pos - 0.05, value, height=0.08, color=color, transform=ax2.transAxes)
    # Value text
    ax2.text(0.95, y_pos, f'{value:.1%}', fontsize=11, fontweight='bold', 
             transform=ax2.transAxes, ha='right', va='center')

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_title('📊 Key Metrics', fontweight='bold', size=11)

# 3. Pie chart for prediction distribution
ax3 = fig.add_subplot(gs[0, 2])
pred_counts = Counter(final_metrics['pred_labels'])
labels_pie = list(pred_counts.keys())
sizes = list(pred_counts.values())
colors_pie = ['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6'][:len(labels_pie)]
explode = [0.05] * len(labels_pie)

ax3.pie(sizes, explode=explode, labels=[l.upper() for l in labels_pie], colors=colors_pie,
        autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
ax3.set_title('🔮 Prediction Distribution', fontweight='bold', size=11)

# 4. Training Loss Curve
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(training_history['epoch'], training_history['train_loss'], 'b-o', linewidth=2, markersize=8)
ax4.fill_between(training_history['epoch'], training_history['train_loss'], alpha=0.3)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Loss')
ax4.set_title('📉 Training Loss', fontweight='bold', size=11)
ax4.grid(True, alpha=0.3)

# 5. Metrics Over Epochs
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(training_history['epoch'], training_history['accuracy'], 'g-o', label='Accuracy', linewidth=2)
ax5.plot(training_history['epoch'], training_history['f1_macro'], 'r-s', label='F1', linewidth=2)
ax5.plot(training_history['epoch'], training_history['rouge1'], 'b-^', label='ROUGE-1', linewidth=2)
ax5.set_xlabel('Epoch')
ax5.set_ylabel('Score')
ax5.set_title('📈 Metrics Progress', fontweight='bold', size=11)
ax5.legend(loc='lower right')
ax5.grid(True, alpha=0.3)
ax5.set_ylim(0, 1)

# 6. Confusion Matrix Heatmap
ax6 = fig.add_subplot(gs[1, 2])
labels = ['yes', 'no', 'maybe']
cm = confusion_matrix(final_metrics['valid_trues'], final_metrics['valid_preds'], labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', xticklabels=labels, yticklabels=labels, 
            ax=ax6, annot_kws={'size': 12, 'weight': 'bold'}, linewidths=0.5, cbar=False)
ax6.set_xlabel('Predicted')
ax6.set_ylabel('True')
ax6.set_title('🔢 Confusion Matrix', fontweight='bold', size=11)

# 7. ROUGE Comparison Bar
ax7 = fig.add_subplot(gs[2, :2])
rouge_metrics_plot = {
    'ROUGE-1\nF1': final_metrics['rouge_avg']['rouge1'],
    'ROUGE-1\nPrecision': final_metrics['rouge_avg']['rouge1_p'],
    'ROUGE-1\nRecall': final_metrics['rouge_avg']['rouge1_r'],
    'ROUGE-2\nF1': final_metrics['rouge_avg']['rouge2'],
    'ROUGE-2\nPrecision': final_metrics['rouge_avg']['rouge2_p'],
    'ROUGE-2\nRecall': final_metrics['rouge_avg']['rouge2_r'],
    'ROUGE-L\nF1': final_metrics['rouge_avg']['rougeL'],
    'ROUGE-L\nPrecision': final_metrics['rouge_avg']['rougeL_p'],
    'ROUGE-L\nRecall': final_metrics['rouge_avg']['rougeL_r'],
}
colors_bar = ['#3498db', '#2980b9', '#1abc9c'] * 3
bars = ax7.bar(rouge_metrics_plot.keys(), rouge_metrics_plot.values(), color=colors_bar, edgecolor='black')
ax7.set_ylabel('Score')
ax7.set_title('📝 Complete ROUGE Breakdown', fontweight='bold', size=11)
ax7.set_ylim(0, 1)
ax7.tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    ax7.annotate(f'{height:.2f}', (bar.get_x() + bar.get_width()/2, height),
                 textcoords="offset points", xytext=(0,3), ha='center', fontsize=8)

# 8. Summary Statistics Text
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')

summary_text = f"""
╔══════════════════════════════════╗
║     📊 FINAL STATISTICS          ║
╠══════════════════════════════════╣
║  Total Samples:    {len(generated_texts):>6}        ║
║  Correct Preds:    {sum(1 for p, t in zip(final_metrics['pred_labels'], final_metrics['true_labels']) if p == t):>6}        ║
║  Unknown Preds:    {final_metrics['pred_labels'].count('unknown'):>6}        ║
╠══════════════════════════════════╣
║  Best ROUGE-1:     {max(rouge_detailed['rouge1']):.3f}        ║
║  Worst ROUGE-1:    {min(rouge_detailed['rouge1']):.3f}        ║
║  Std ROUGE-1:      {np.std(rouge_detailed['rouge1']):.3f}        ║
╠══════════════════════════════════╣
║  LoRA Rank:        {rank:>6}        ║
║  LoRA Alpha:       {alpha:>6}        ║
║  Epochs:           {num_epochs:>6}        ║
╚══════════════════════════════════╝
"""
ax8.text(0.5, 0.5, summary_text, transform=ax8.transAxes, fontsize=10,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='#f8f9fa', edgecolor='#dee2e6'))

plt.tight_layout()
plt.savefig('evaluation_dashboard.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n💾 Dashboard saved as 'evaluation_dashboard.png'")

In [ ]:
# ============================================================================
# 🧪 CUSTOM INFERENCE EXAMPLES
# ============================================================================

enable_lora_forward(model)
model.eval()

test_questions = [
    "Question: Does aspirin reduce heart attack risk? Instruction: Answer yes, no, or maybe, then explain briefly.",
    "Question: Is water wet? Instruction: Answer in one sentence.",
    "Question: Is vitamin C a cure for cancer? Answer yes, no, or maybe.",
    "Question: Does aspirin reduce fever? Answer yes, no, or maybe.",
    "Question: Can exercise help prevent diabetes? Answer yes, no, or maybe, then explain.",
]

print("\n" + "="*80)
print("🧪 CUSTOM INFERENCE EXAMPLES")
print("="*80)

fig, axes = plt.subplots(len(test_questions), 1, figsize=(14, 3*len(test_questions)))

for i, question in enumerate(test_questions):
    inputs = tokenizer(question, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100)
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract answer
    short_answer = extract_short_answer(response)
    
    print(f"\n{'─'*80}")
    print(f"📝 Question {i+1}:")
    print(f"   {question}")
    print(f"\n🤖 Model Response:")
    print(f"   {response}")
    print(f"\n🏷️ Extracted Answer: {short_answer.upper()}")
    
    # Visualize in subplot
    ax = axes[i] if len(test_questions) > 1 else axes
    ax.axis('off')
    
    # Color based on answer type
    colors = {'yes': '#27ae60', 'no': '#e74c3c', 'maybe': '#f39c12', 'unknown': '#95a5a6'}
    bg_color = colors.get(short_answer, '#95a5a6')
    
    # Create text box
    q_text = question[:80] + "..." if len(question) > 80 else question
    r_text = response[:150] + "..." if len(response) > 150 else response
    
    ax.text(0.02, 0.7, f"Q: {q_text}", fontsize=10, fontweight='bold', 
            transform=ax.transAxes, wrap=True, verticalalignment='top')
    ax.text(0.02, 0.35, f"A: {r_text}", fontsize=9, 
            transform=ax.transAxes, wrap=True, verticalalignment='top', style='italic')
    
    # Answer badge
    ax.add_patch(plt.Rectangle((0.85, 0.4), 0.12, 0.4, transform=ax.transAxes, 
                                facecolor=bg_color, edgecolor='black', linewidth=2))
    ax.text(0.91, 0.6, short_answer.upper(), fontsize=11, fontweight='bold', color='white',
            transform=ax.transAxes, ha='center', va='center')

plt.tight_layout()
plt.show()

disable_lora_forward(model)
print(f"\n{'='*80}")

Custom Inference Examples:

Q: Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences.
A: Answer: yes. Explanation: Aspirin reduces the risk of heart attack and can be taken as a supplement.
----------------------------------------

Q: Question: Is water wet? Instruction: Answer in one sentence.
A: Answer: yes. Explanation: The water is wet.
----------------------------------------

Q: Question: Is vitamin C a cure for cancer? Answer yes or no.
A: Answer: yes. Explanation: Vitamin C is a powerful antioxidant that can be used as a treatment for cancer.
----------------------------------------

Q: Question: Does aspirin reduce fever? Answer yes or no.
A: Answer: yes. Explanation: Aspirin reduces fever, but it does not reduce the risk of a cold.
----------------------------------------


In [ ]:
# ============================================================================
# 💾 MERGE LORA & SAVE MODEL
# ============================================================================

# Merge LoRA weights into base model
merge_lora_to_linear(model)
model.eval()

print("✅ LoRA weights merged into base model.")

# Final parameter count after merge
total_final = sum(p.numel() for p in model.parameters())
print(f"\n📊 Final Model Statistics:")
print(f"   Total Parameters: {total_final:,}")
print(f"   Model Size (approx): {total_final * 4 / 1024 / 1024:.2f} MB (FP32)")

LoRA weights merged into base model.


In [ ]:
# ============================================================================
# 💾 SAVE MODEL (OPTIONAL)
# ============================================================================

# Uncomment the following lines to save the model
# save_path = "./flan-t5-base-pubmedqa-lora-merged"
# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)
# print(f"✅ Model saved to: {save_path}")

# ============================================================================
# 🎉 TRAINING COMPLETE - FINAL SUMMARY
# ============================================================================

print("\n" + "🎉"*30)
print("\n" + "="*80)
print("🏆 TRAINING & EVALUATION COMPLETE!")
print("="*80)
print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                           📋 FINAL SUMMARY                                   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Model:              {MODEL_NAME:<40}             ║
║  Training Method:    LoRA (Low-Rank Adaptation)                              ║
║  LoRA Rank:          {rank:<10} Alpha: {alpha:<10}                           ║
║  Epochs Trained:     {num_epochs:<10}                                                 ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                       🎯 CLASSIFICATION METRICS                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Accuracy:           {final_metrics['accuracy']:.2%}                                                   ║
║  F1-Score (Macro):   {final_metrics['f1_macro']:.2%}                                                   ║
║  Precision (Macro):  {final_metrics['precision_macro']:.2%}                                                   ║
║  Recall (Macro):     {final_metrics['recall_macro']:.2%}                                                   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                       📝 LONG-ANSWER QUALITY                                 ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ROUGE-1 (F1):       {final_metrics['rouge_avg']['rouge1']:.2%}                                                   ║
║  ROUGE-2 (F1):       {final_metrics['rouge_avg']['rouge2']:.2%}                                                   ║
║  ROUGE-L (F1):       {final_metrics['rouge_avg']['rougeL']:.2%}                                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")
print("🎉"*30 + "\n")